# cv_r1 学習フェーズ【Colab・要 GPU・Blackwell / 標準 GPU 両対応版】

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slp-hu/Style-Bert-VITS2/blob/layer-b-cadence-seq/colab/cv_r1_train_colab.ipynb)

setup（環境構築）の後に実行する学習ノート。**bert_gen → style_gen → 学習**をこの1本で回し、
実行結果（ログ）をセルに残す。トラブル時はこのノートの出力を見れば切り分けできる。

Colab が割り当てる GPU に応じて **§2 が経路を自動分岐**する:

| 割り当て GPU | torch | ランタイム再起動 | shim (§3) |
|---|---|---|---|
| T4 / L4 / A100 等（sm_90 以下） | 2.3.1+cu121（requirements の pin のまま） | 不要 | 不要（自動スキップ） |
| **Blackwell 系**（RTX PRO 6000 = sm_120 等） | **2.11.0+cu128 へ入替** | **§2 の後に必須** | **必須** |

**実行手順（上から順に）**
1. §1 → §2 を実行
2. §2 が「ランタイム再起動が必須」と表示したら（= Blackwell）: **[ランタイム] → [セッションを再起動]**
   → **§1 から再実行**（§2 は入替をスキップして検証だけ通る）→ §3 へ
3. §3（shim）→ §4（ローカル展開）→ §5（検証ゲート）→ §6 bert_gen → §7 style_gen → §8 学習

**データはローカル運用が既定（推奨）**: Drive 直読み学習は小ファイル I/O 律速で GPU 使用率 0% に
張り付き、完走に丸1日超かかる。§4 で **Zenodo から直接ローカル `/content` に展開**し、
checkpoint / model_assets だけ Drive へ symlink して永続化する（読み=ローカルで速く、保存=Drive で再開可）。
全 GPU で有効、Blackwell では実質必須。ローカル運用では `cv_r1_deploy_colab.ipynb`（Drive への展開）は
**実行不要**（Drive 上にデータ一式を常設したい場合のみ任意で使用）。

**前提**
- `cv_r1_train_setup_colab.ipynb` 実行済み（Drive に fork clone・initialize 済み）
- **GPU ランタイム**（[ランタイム]→[ランタイムのタイプを変更]→GPU）

**このノートでやらないこと**
- `preprocess_text` は**走らせない**（esd 配置済み・spk2id を再生成し得る＝emb_g 行ズレの原因）


In [ ]:
# ===== §1 マウント・前提チェック【毎回・ランタイム再起動後も最初に実行】=====
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, subprocess

DRIVE_BASE = Path("/content/drive/MyDrive/Style-Bert-VITS2")   # setup と同じ値（clone の置き場所）
assert DRIVE_BASE.exists(), f"★Drive に fork clone が無い: {DRIVE_BASE}（先に cv_r1_train_setup_colab.ipynb を実行）"
os.chdir(DRIVE_BASE); print("cwd:", Path.cwd())   # 依存インストールは requirements.txt をここから読む（後段でローカルへ移る）

br = subprocess.run(["git","rev-parse","--abbrev-ref","HEAD"], capture_output=True, text=True).stdout.strip()
assert br == "layer-b-cadence-seq", f"★branch が違う: {br} → git checkout layer-b-cadence-seq"
print("branch:", br)

g = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
assert g.returncode == 0 and "GPU" in g.stdout, "★GPU ランタイムでない → [ランタイム]→[ランタイムのタイプを変更]→GPU"
print(g.stdout.strip())


## §1.5 スモーク学習（動作確認）設定 — 無料 Colab 用

**目的はパイプラインの動作確認**（環境構築 → bert_gen → style_gen → warm-start 学習 → checkpoint 保存が
通ること）。数百 step の学習に品質的な意味はない。本番学習では `SMOKE = False` のまま触らない。

- スモークは全量 4.81 GiB ではなく**専用サブセットバンドル**（`cadence_cv_r1_smoke_v1.tgz`,
  30 話者 × train 40 発話 ≒ 1,200 行 + val 60 行, ~0.3 GB）を GitHub Releases から取得する。
  DL は 1〜2 分。既定の batch 4 / 2 epoch ≒ 600 step（T4 実測 ~1.8 s/it ≒ 学習 20 分弱）で、**T4 で全体 40 分前後**に収まり、無料枠のセッション上限に十分収まる。
- 無料枠の T4 (sm_75) は**標準経路**（torch 2.3.1・ランタイム再起動なし・shim 不要）を通る。
- checkpoint はスモーク専用のローカルツリー `Data/cv_r1_smoke/models/` に保存され、
  **Drive の本番 `models/` には一切書かない**（本番 ckpt からの誤再開も起きない）。
  model_assets 側は `cv_r1_smoke/` サブフォルダに分離される。
- config の spk2id（298 話者）はバンドルでもそのまま維持 → モデル形状・warm-start の検証は本番と等価。
- T4 など compute capability < 8.0 の GPU では bf16/fp16 を自動で無効化する（fp32）。
- サブセットの内容はバンドル側で固定（生成スクリプト: `colab/make_smoke_bundle.py`）。
  ここで決めるのは学習の回し方だけ。


In [ ]:
# ===== §1.5 スモーク学習（動作確認）設定 =====
SMOKE = False            # ← 動作確認するときだけ True（本番学習は False のまま）
SMOKE_EPOCHS        = 2
SMOKE_BATCH         = 4    # T4 (16GB) の fp32 実測で 8 は OOM（WavLM 判別器が重い）。それでも溢れたら 2 に
SMOKE_EVAL_INTERVAL = 100  # 保存間隔（既定 1000 のままだと保存前に学習が終わる）
print("SMOKE =", SMOKE)


In [ ]:
# ===== §2 GPU 判定と依存インストール【毎セッション実行・再起動後も再実行（冪等）】=====
# GPU の compute capability で経路を自動分岐する:
#   ・sm_90 以下（T4/L4/A100 等）: requirements の pin どおり torch 2.3.1(cu121)。再起動不要。
#   ・sm_100 以上（Blackwell 系: RTX PRO 6000 = sm_120 等）: torch 2.3.1 は sm_90 までで非対応
#     （GPU forward で落ちる）→ torch 2.11.0+cu128 へ入替（07a 方式）。★入替後はランタイム再起動が必須。
# 再起動後にこのセルを再実行すると、入替をスキップして検証だけ行う。
import subprocess, sys, re, os
from pathlib import Path

def _run(cmd, stream=False):
    print("$", " ".join(cmd))
    if stream:   # 進捗をそのまま流す（★torch の数GB DL は -q だと無言=フリーズと誤認するため）
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout: print(line, end="")
        p.wait(); assert p.returncode == 0, f"失敗: {cmd}"
    else:
        r = subprocess.run(cmd, capture_output=True, text=True)
        print((r.stdout or "")[-1200:] or "(quiet)")
        if r.returncode != 0:
            print(r.stderr[-4000:]); raise SystemExit(f"失敗: {cmd}")

# --- GPU capability（torch を import せず nvidia-smi で判定するのが肝）---
cap = subprocess.run(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()
assert cap and cap[0], "★GPU が見えない（GPU ランタイムか確認）"
CAP = float(cap[0]); IS_BLACKWELL = CAP >= 10.0
print(f"compute capability: {cap[0]} (sm_{int(CAP*10)}) → 経路: "
      + ("Blackwell（torch 2.11+cu128 入替）" if IS_BLACKWELL else "標準（torch 2.3.1 のまま）"))

# --- 現在の torch を subprocess で確認（in-kernel import しない = 再起動不要性を保つ）---
q = subprocess.run([sys.executable,"-c","import torch;print(torch.__version__)"], capture_output=True, text=True)
TORCH_NOW = q.stdout.strip()
print("現在の torch:", TORCH_NOW or "(未導入)")
ALREADY_SWAPPED = IS_BLACKWELL and TORCH_NOW.startswith("2.11.")

if ALREADY_SWAPPED:
    print("→ 再起動後の再実行と判断: torch 入替済みのため install をスキップ")
else:
    # (1) requirements から faster-whisper を除外して install（07a 方式）。
    #     faster-whisper==0.10.1 が av==10.* をソースビルドしようとし Py3.12 で失敗するため。
    #     文字起こし用途で cv_r1（bert_gen/style_gen/train/eval）には不要。
    req = Path("requirements.txt"); req_train = Path("requirements_no_whisper.txt")
    _drop = re.compile(r"^(faster-whisper|av)==")   # ビルドで詰まるパッケージが増えたらここに追加（例: stable_ts）
    kept = [l for l in req.read_text(encoding="utf-8").splitlines() if not _drop.match(l.strip())]
    req_train.write_text("\n".join(kept) + "\n", encoding="utf-8")
    _run([sys.executable,"-m","pip","install","-q","-r",str(req_train)])

    # (2) Blackwell のみ: torch 2.11.0+cu128 へ入替（実績のある手順をそのまま踏む）
    if IS_BLACKWELL:
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torch","torchaudio","torchvision","torchcodec","torchao","torchtune","torchdata"])
        _run([sys.executable,"-m","pip","install","torch==2.11.0","torchaudio==2.11.0",
              "--index-url","https://download.pytorch.org/whl/cu128"], stream=True)   # ★-q 禁止
        _run([sys.executable,"-m","pip","install","-q","soundfile"])
        _run([sys.executable,"-m","pip","uninstall","-y",
              "torchcodec","torchvision","torchao","torchtune","torchdata"])

    # (3) HF スタック固定（transformers 未ピン → Colab 既定の transformers 5.x が torch>=2.4 を要求して
    #     torch を無効化し、bert_gen が "AutoModelForMaskedLM requires PyTorch" で落ちる問題の恒久対策）
    _run([sys.executable,"-m","pip","install","-q",
          "transformers==4.41.2","huggingface_hub==0.23.5","tokenizers<0.20",
          "pytorch-lightning==2.2.5","torchmetrics<1.5","pyannote.audio==3.1.1",
          "scipy==1.13.1","numpy==1.26.4"])

# --- 検証（別プロセス。transformers から torch が見えているかまで確認）---
v = subprocess.run([sys.executable,"-c",
    "import torch;from transformers.utils import is_torch_available;"
    "print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),"
    "'| transformers_sees_torch',is_torch_available())"], capture_output=True, text=True)
print(v.stdout.strip() or v.stderr[-800:])
assert "transformers_sees_torch True" in v.stdout, "★transformers が torch を認識していない → このセルをやり直す"
if (not IS_BLACKWELL) or ALREADY_SWAPPED:
    assert "cuda True" in v.stdout, "★CUDA が使えない（GPU ランタイム / torch ビルドを確認）"

if IS_BLACKWELL and not ALREADY_SWAPPED:
    print()
    print("=" * 70)
    print("★torch を入れ替えた → ここで【ランタイム再起動】が必須★")
    print("  [ランタイム] → [セッションを再起動] のあと、§1 → §2 → §3 の順に再実行してから先へ進む。")
    print("  （再起動で cwd も torch 状態もリセットされる。最初のセルを飛ばさないこと）")
    print("=" * 70)


In [ ]:
# ===== §3 torchaudio / torch.load shim【Blackwell 経路のみ実体化・毎セッション実行】=====
# torch 2.11 系では torchaudio.set_audio_backend が削除され、torchaudio.load も torchcodec 経由で壊れる。
# pyannote.audio が import 時に set_audio_backend を呼ぶため、shim なしでは style_gen / 合成が落ちる。
# `!python` で走る bert_gen / style_gen / train は【別プロセス】なので、カーネル内 monkeypatch では効かない
# → sitecustomize.py + PYTHONPATH で全 Python プロセスに注入する（ここが肝）。
import os, subprocess, sys
from pathlib import Path

if not IS_BLACKWELL:
    print("標準経路（torch 2.3.1）: shim 不要 → スキップ")
else:
    compat = Path("/content/_compat"); compat.mkdir(exist_ok=True)
    shim = compat / "sitecustomize.py"
    SHIM_SRC = '# sitecustomize: torch 2.11 環境の互換 shim（cv_r1 公開ノート用）\n# 1) torch.load の weights_only 既定を False に戻す（旧 ckpt / torch.hub モデルの読込互換）\n# 2) torchaudio.set_audio_backend / get_audio_backend を復活（pyannote.audio の import 時呼び出し対策）\n# 3) torchaudio.load / info を soundfile 実装に置換（torchcodec 経由の破綻を回避）\ntry:\n    import torch\n    _orig_torch_load = torch.load\n    def _patched_torch_load(*args, **kwargs):\n        kwargs.setdefault("weights_only", False)\n        return _orig_torch_load(*args, **kwargs)\n    torch.load = _patched_torch_load\nexcept Exception:\n    pass\n\ntry:\n    import torchaudio\n\n    def _noop_set_backend(*args, **kwargs):\n        return None\n    def _get_backend(*args, **kwargs):\n        return "soundfile"\n    torchaudio.set_audio_backend = _noop_set_backend\n    torchaudio.get_audio_backend = _get_backend\n\n    def _sf_load(filepath, frame_offset=0, num_frames=-1, normalize=True,\n                 channels_first=True, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        import torch as _torch\n        frames = int(num_frames) if int(num_frames) > 0 else -1\n        data, sr = sf.read(str(filepath), start=int(frame_offset), frames=frames,\n                           dtype="float32", always_2d=True)\n        wav = _torch.from_numpy(data.T if channels_first else data)\n        return wav, sr\n    torchaudio.load = _sf_load\n\n    def _sf_info(filepath, format=None, buffer_size=4096, backend=None):\n        import soundfile as sf\n        info = sf.info(str(filepath))\n        class _AudioMetaData:\n            pass\n        meta = _AudioMetaData()\n        meta.sample_rate = info.samplerate\n        meta.num_frames = info.frames\n        meta.num_channels = info.channels\n        meta.bits_per_sample = 16\n        meta.encoding = "PCM_S"\n        return meta\n    torchaudio.info = _sf_info\nexcept Exception:\n    pass\n'
    shim.write_text(SHIM_SRC, encoding="utf-8")

    # 書き出し事故（末尾のエスケープ崩れ等 → SyntaxError）を必ず py_compile で検証する
    r = subprocess.run([sys.executable,"-m","py_compile",str(shim)], capture_output=True, text=True)
    assert r.returncode == 0, "★shim が SyntaxError: " + r.stderr[-600:]

    pp = os.environ.get("PYTHONPATH","")
    if str(compat) not in pp.split(":"):
        os.environ["PYTHONPATH"] = f"{compat}:{pp}" if pp else str(compat)
    print("PYTHONPATH:", os.environ["PYTHONPATH"])

    # 機能確認: 別プロセスで torchaudio.load が shim（_compat）実装に置換されているか
    chk = subprocess.run([sys.executable,"-c",
        "import torchaudio;torchaudio.set_audio_backend('soundfile');"
        "import inspect;print('shim OK:', inspect.getsourcefile(torchaudio.load))"],
        capture_output=True, text=True, env=os.environ.copy())
    print(chk.stdout.strip() or chk.stderr[-800:])
    assert "shim OK" in chk.stdout and "_compat" in chk.stdout, "★shim が別プロセスに効いていない"


## §4 ローカル運用への展開

fork コードを Drive clone からローカルへ rsync（`Data` / `.git` 除外 = 小ファイル地獄を回避、
`bert/` `slm/` `pretrained_jp_extra/` は少数の大ファイルなので Drive 読みでも数分）、
データは Zenodo からローカルに直接展開、checkpoint / model_assets は Drive へ symlink する。
あわせて fork 固有の `default_style.py` バグ修正と、底モデル（warm-start 元）の配置も行う。


In [ ]:
# ===== §4-A fork コードをローカルへ + default_style.py パッチ =====
import os, shutil, subprocess
from pathlib import Path

ROOT = Path("/content/Style-Bert-VITS2")     # 学習実行ルート（ローカル）
DATA = ROOT / "Data" / "cv_r1"

# fork コードをローカルへ（初回 数分。Data / .git / model_assets / eval_out は除外）
!rsync -a --info=progress2 --exclude Data --exclude .git --exclude model_assets --exclude eval_out {DRIVE_BASE}/ {ROOT}/
os.chdir(ROOT); print("cwd:", Path.cwd())
assert (ROOT/"train_ms_jp_extra.py").exists(), "★rsync 失敗（DRIVE_BASE を確認）"

# default_style.py の cadseq 巻き込みバグ修正（この fork 固有。upstream には無い）:
# save_neutral_vector / save_styles_by_dirs が rglob("*.npy") で cadence sidecar
# （*.cadseq.npy, 形 (P,32)）まで拾い、256 次元 style と混ざって学習開始直後に
# 「ValueError: dimension 1 ... size 25 vs 40」で落ちる。→ 3箇所に除外ガードを入れる。
def patch_default_style(path):
    p = Path(path)
    src = p.read_text(encoding="utf-8")
    if src.count("cadseq") >= 3:
        print("パッチ適用済み:", p); return
    src = src.replace(
        'for file in wav_dir.rglob("*.npy"):',
        'for file in wav_dir.rglob("*.npy"):\n'
        '        if file.name.endswith(".cadseq.npy"):\n'
        '            continue')
    src = src.replace(
        'npy_files = list(style_dir.rglob("*.npy"))',
        'npy_files = [f for f in style_dir.rglob("*.npy") if not f.name.endswith(".cadseq.npy")]')
    p.write_text(src, encoding="utf-8")
    n = p.read_text(encoding="utf-8").count("cadseq")
    assert n == 3, f"★パッチ結果が想定外（cadseq 出現 {n} ≠ 3）: {p}"
    import py_compile; py_compile.compile(str(p), doraise=True)
    print("パッチ適用:", p, "(cadseq 出現 = 3)")

patch_default_style(ROOT / "default_style.py")         # ローカル（学習が読む方）
patch_default_style(DRIVE_BASE / "default_style.py")   # Drive clone にも当てて永続化（次回 rsync で戻らないように）


In [ ]:
# ===== §4-B データ取得（→ ローカル展開）=====
import json, shutil
from pathlib import Path
# 全量: Zenodo の 2 tar を VM ローカルに直接 DL・展開（Drive 直読みの小ファイル I/O 律速を回避）。
# スモーク: 専用サブセットバンドル（~0.3 GB）を GitHub Releases から取得。
DOI_REC = "https://zenodo.org/records/21119791/files"
META   = "cadence_cv_r1_meta_v20260702.tgz"    # 31.9 MiB: config / esd / cadseq / MANIFEST 等
WAVS   = "cadence_cv_r1_wavs_v20260702.tar"    # 4.81 GiB 無圧縮: wav 18015 (sr44100 mono)
SMOKEB = "cadence_cv_r1_smoke_v1.tgz"          # 30 話者×40 発話サブセット（wav+cadseq+esd+config）
SMOKE_URL = "https://github.com/slp-hu/Style-Bert-VITS2/releases/download/cv_r1-smoke-v1/" + SMOKEB
DL = Path("/content/_zenodo"); DL.mkdir(exist_ok=True)
targets = ([(SMOKEB, SMOKE_URL)] if SMOKE else
           [(META, f"{DOI_REC}/{META}?download=1"), (WAVS, f"{DOI_REC}/{WAVS}?download=1")])
MIN_SIZE = {META: 30_000_000, WAVS: 5_000_000_000, SMOKEB: 100_000_000}   # 完全性の下限

def n_files(d, pat): return sum(1 for _ in Path(d).rglob(pat)) if Path(d).exists() else 0
def _placed():
    if not (DATA/"config.json").exists(): return False
    man = DATA/"SMOKE_MANIFEST.json"
    if SMOKE:
        return man.exists() and n_files(DATA, "*.wav") == json.load(open(man, encoding="utf-8"))["wav_count"]
    return (not man.exists()) and n_files(DATA, "*.wav") == 18015   # スモーク残骸なら全量を取り直す

if _placed():
    print("配置済み → DL / 展開をスキップ")
else:
    # --- 取得: Drive キャッシュ → aria2 16並列 → wget フォールバック ---------------
    # Zenodo は単一接続だと 1〜2 MB/s まで落ちることがある（4.8 GiB で 1 時間超）。
    # aria2 の並列レンジ DL で通常 5〜15 倍出る。DRIVE_BASE/zenodo_cache/ に tar を
    # 置いておけば取得元を経由せず Drive から複写する（単一大ファイルなので速い）。
    CACHE = DRIVE_BASE / "zenodo_cache"
    CACHE_TO_DRIVE = False      # True: DL 成功後に Drive へ保存（次回以降 DL 不要。全量は約 5 GB 消費）
    def _ok(p, name): return p.exists() and p.stat().st_size >= MIN_SIZE[name]
    def _fetch(name, url):
        dst = DL / name
        if _ok(dst, name):
            print("取得済み:", name); return
        if dst.exists(): dst.unlink()   # 不完全ファイルは捨てる（低速 DL の再開より並列 DL のほうが速い）
        if _ok(CACHE / name, name):
            print("Drive キャッシュから複写:", name)
            shutil.copy2(CACHE / name, dst); return
        if not shutil.which("aria2c"):
            !apt-get -qq -y install aria2 > /dev/null
        !aria2c -x16 -s16 -k1M --console-log-level=warn --summary-interval=15 -d {DL} -o {name} "{url}"
        if not _ok(dst, name):
            print("★aria2 失敗 → wget にフォールバック")
            !wget -c -O {DL}/{name} "{url}"
        assert _ok(dst, name), f"★DL 不完全: {name} (size={dst.stat().st_size if dst.exists() else 0})"
    for name, url in targets:
        _fetch(name, url)
    if CACHE_TO_DRIVE:
        CACHE.mkdir(exist_ok=True)
        for name, _ in targets:
            if not _ok(CACHE / name, name):
                print("Drive へキャッシュ保存:", name); shutil.copy2(DL / name, CACHE / name)
    # --- 展開 → 配置（config.json の位置からデータセットルートを自動判定）---------
    stage = Path("/content/_zenodo/x")
    if stage.exists(): shutil.rmtree(stage)
    stage.mkdir(parents=True)
    for name, _ in targets:
        print("展開:", name)
        !tar -xf {DL}/{name} -C {stage}
    cfgs = list(stage.rglob("config.json"))
    assert len(cfgs) == 1, f"★config.json の位置を特定できない: {cfgs}"
    src_root = cfgs[0].parent; print("データセットルート検出:", src_root)
    DATA.parent.mkdir(parents=True, exist_ok=True)
    if DATA.exists() and not DATA.is_symlink(): shutil.rmtree(DATA)
    shutil.move(str(src_root), str(DATA))
    if n_files(DATA, "*.wav") == 0:
        # wav tar のルート prefix が meta と異なる場合: 残りを DATA 直下へ統合
        for child in list(stage.iterdir()):
            print("統合:", child.name, "->", DATA/child.name)
            shutil.move(str(child), str(DATA/child.name))

# --- 期待値の設定（§5 検証ゲート・§6 bert_gen が参照）---------------------------
if SMOKE:
    _man = json.load(open(DATA/"SMOKE_MANIFEST.json", encoding="utf-8"))
    EXP_WAV, EXP_CADSEQ = _man["wav_count"], _man["cadseq_count"]
else:
    EXP_WAV, EXP_CADSEQ = 18015, 11010
print(f"wav: {n_files(DATA, '*.wav')} ({EXP_WAV} が正) / cadseq: {n_files(DATA, '*.cadseq.npy')} ({EXP_CADSEQ} が正)")


In [ ]:
# ===== §4-C checkpoint / model_assets の Drive 永続化 + 底モデル配置 =====
import os, shutil
from pathlib import Path

# (1) checkpoint 置き場を Drive へ symlink（読み=ローカル / 保存=Drive。切断後も最新 ckpt から再開できる）
drive_models = DRIVE_BASE/"Data"/"cv_r1"/"models"; drive_models.mkdir(parents=True, exist_ok=True)
local_models = DATA/"models"
if local_models.exists() and not local_models.is_symlink(): shutil.rmtree(local_models)
!ln -sfn {drive_models} {local_models}
print("models       ->", os.path.realpath(local_models))

# (2) 推論用 model_assets（学習末尾の safetensors 書き出し先）も Drive へ symlink
drive_assets = DRIVE_BASE/"model_assets"; drive_assets.mkdir(exist_ok=True)
local_assets = ROOT/"model_assets"
if local_assets.exists() and not local_assets.is_symlink(): shutil.rmtree(local_assets)
!ln -sfn {drive_assets} {local_assets}
print("model_assets ->", os.path.realpath(local_assets))

# (3) 底モデル（warm-start 元）を models/ へ配置。
#     train は Data/cv_r1/models/{G,D,WD}_0.safetensors を warm-start 元として読む。
#     initialize.py は pretrained_jp_extra/ に置くだけなので、コピーしないと
#     ★scratch 学習になり cadence 設計が壊れる（freeze された decoder が初期値のまま凍結される）。
#     既に G_*.pth がある場合は「再開」なのでコピーしない
#     （学習が step 保存時に *_0.safetensors を「古い ckpt」として消すのは正常挙動。原本は pretrained_jp_extra/ に残る）。
resumable = sorted(drive_models.glob("G_*.pth"))
if resumable:
    print("既存 checkpoint あり → 再開モード（底モデルコピー不要）:", resumable[-1].name)
else:
    for f in ("G_0.safetensors","D_0.safetensors","WD_0.safetensors"):
        src = DRIVE_BASE/"pretrained_jp_extra"/f
        assert src.exists(), f"★{src} が無い（setup §3 initialize を先に実行）"
        shutil.copy(src, drive_models/f); print("底モデル配置:", f)


## §4.5 スモーク設定の適用

`SMOKE` の値に応じて config と checkpoint 出力先を切り替える。**SMOKE = False でもこのセルは必ず実行する**
（config の原本復元と検証ゲート用の期待値設定を兼ねる）。esd リストはスモークバンドルに同梱の
サブセット済みのものをそのまま使う（このセルでは加工しない）。config の原本は初回に
`config.full.json` へ退避し、以後は常に原本から再構成するので何度実行しても安全（冪等）。
True ↔ False を切り替えたときは **§4-B から**実行し直す（データ一式が入れ替わる）。


In [ ]:
# ===== §4.5 スモーク設定の適用 =====
import json, re, shutil, subprocess, math
from pathlib import Path

FULL_TRAIN, FULL_VAL = 14226, 3789
cfgp, cbk = DATA/"config.json", DATA/"config.full.json"
if not cbk.exists(): shutil.copy2(cfgp, cbk)   # 原本退避（初回のみ）。以後は原本から再構成
cyml, cyml_bk = ROOT/"config.yml", ROOT/"config.yml.orig"
if not cyml.exists():
    # config.yml は SBV2 の config.py が初回 import 時に default_config.yml から自動生成する。
    # このセルはどの SBV2 スクリプトより先に走るため、無ければここで同じことをしておく。
    shutil.copy2(ROOT/"default_config.yml", cyml)
if not cyml_bk.exists(): shutil.copy2(cyml, cyml_bk)   # config.yml も原本退避（同上）

if not SMOKE:
    shutil.copy2(cbk, cfgp); shutil.copy2(cyml_bk, cyml)
    SMOKE_EXP_TRAIN, SMOKE_EXP_VAL = FULL_TRAIN, FULL_VAL
    MDIR = "Data/cv_r1"
    print("SMOKE=False → 全量学習（config.json / config.yml を原本に復元）")
else:
    man = json.load(open(DATA/"SMOKE_MANIFEST.json", encoding="utf-8"))
    SMOKE_EXP_TRAIN, SMOKE_EXP_VAL = man["train_lines"], man["val_lines"]

    # --- config 上書き（spk2id は 298 のまま = 底モデル・embedding 形状に手を触れない）---
    cfg = json.load(open(cbk, encoding="utf-8"))
    cfg["train"]["epochs"]        = SMOKE_EPOCHS
    cfg["train"]["batch_size"]    = SMOKE_BATCH
    cfg["train"]["eval_interval"] = SMOKE_EVAL_INTERVAL
    cfg["train"]["log_interval"]  = 20
    if "model_name" in cfg: cfg["model_name"] = "cv_r1_smoke"   # model_assets を本番と分離
    cap = float(subprocess.run(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],
                               capture_output=True, text=True).stdout.strip().splitlines()[0])
    if cap < 8.0:  # Ampere 未満 (T4=7.5 等) は bf16 非対応 → fp32 に固定
        cfg["train"]["bf16_run"] = False; cfg["train"]["fp16_run"] = False
        print(f"compute capability {cap} < 8.0 → bf16/fp16 を無効化 (fp32)")
    json.dump(cfg, open(cfgp, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    # --- checkpoint 先をスモーク専用ローカルツリーに分離（Drive の本番 models/ を汚さない）---
    MDIR = "Data/cv_r1_smoke"
    # train は -m ディレクトリ直下の wavs/ を直接見る（default_style.save_styles_by_dirs）
    # → データ実体へ symlink（バンドルの wav + style_gen の npy をそのまま参照させる）
    wl = ROOT/MDIR/"wavs"
    wl.parent.mkdir(parents=True, exist_ok=True)
    if not wl.is_symlink():
        if wl.exists(): shutil.rmtree(wl)
        wl.symlink_to(DATA/"wavs")
    # 成果物の分離は config.yml の model_name で決まる（out_dir = model_assets/{model_name},
    # safetensors 命名も同じ。config.json の model_name では変わらない）→ 原本から書き替え
    t = re.sub(r"(?m)^model_name:.*$", 'model_name: "cv_r1_smoke"',
               cyml_bk.read_text(encoding="utf-8"), count=1)
    assert 'model_name: "cv_r1_smoke"' in t, "★config.yml に model_name 行が見つからない"
    cyml.write_text(t, encoding="utf-8")
    # 学習済み話者リストを assets 側へ（合成ノートが「有効な30話者」を知るため。Drive に永続）
    aout = ROOT/"model_assets"/"cv_r1_smoke"; aout.mkdir(parents=True, exist_ok=True)
    spks = sorted({l.split("|")[1] for l in open(DATA/"esd_train.list", encoding="utf-8")})
    json.dump(spks, open(aout/"trained_speakers.json", "w", encoding="utf-8"), ensure_ascii=False)
    sm = ROOT/MDIR/"models"; sm.mkdir(parents=True, exist_ok=True)
    if not any(sm.glob("G_*.pth")):
        for f in ("G_0.safetensors", "D_0.safetensors", "WD_0.safetensors"):
            shutil.copy2(ROOT/"pretrained_jp_extra"/f, sm/f)
        print("底モデルを", sm, "へ配置")

    steps = math.ceil(SMOKE_EXP_TRAIN/SMOKE_BATCH) * SMOKE_EPOCHS
    assert steps >= SMOKE_EVAL_INTERVAL, (
        f"★総 step ({steps}) が保存間隔 ({SMOKE_EVAL_INTERVAL}) 未満: checkpoint が一度も保存されない。"
        "SMOKE_EVAL_INTERVAL を下げる")
    print(f"スモーク: 話者 {man['n_speakers']} / train {SMOKE_EXP_TRAIN} 行 / val {SMOKE_EXP_VAL} 行 "
          f"/ batch {SMOKE_BATCH} → 総 {steps} step（{SMOKE_EVAL_INTERVAL} step ごと保存）")


In [ ]:
# ===== §5 検証ゲート（全 OK になってから先へ進む）=====
import json
from pathlib import Path

def n_files(d, pat): return sum(1 for _ in Path(d).rglob(pat)) if Path(d).exists() else 0
def _lines(p): return sum(1 for _ in open(p, encoding="utf-8"))
cfg = json.load(open(DATA/"config.json", encoding="utf-8"))
exp_tr, exp_va = SMOKE_EXP_TRAIN, SMOKE_EXP_VAL          # §4.5 が設定（全量時 14226 / 3789）
ckpt_dir = (ROOT/MDIR/"models") if SMOKE else (DATA/"models")

checks = [
    ("config: spk2id = 298 話者 (cv_0001..cv_0298)", len(cfg["data"].get("spk2id", {})) == 298),
    ("config: freeze_decoder = True",                 cfg["train"].get("freeze_decoder") is True),
    ("config: 相対パス (Data/cv_r1/…)",               str(cfg["data"]["training_files"]).startswith("Data/cv_r1/")),
    (f"esd_train.list = {exp_tr} 行",                 _lines(DATA/"esd_train.list") == exp_tr),
    (f"esd_val.list   = {exp_va} 行",                 _lines(DATA/"esd_val.list") == exp_va),
    (f"wav = {EXP_WAV}",                              n_files(DATA, "*.wav") == EXP_WAV),
    (f"cadseq sidecar = {EXP_CADSEQ}",                n_files(DATA, "*.cadseq.npy") == EXP_CADSEQ),
    ("models/ が Drive への symlink",                 (DATA/"models").is_symlink()),
    (f"底モデル or 再開 ckpt あり ({ckpt_dir})",      any(Path(ckpt_dir).glob("G_*"))),
    ("model_assets/ が Drive への symlink",           (ROOT/"model_assets").is_symlink()),
    ("default_style.py パッチ (cadseq 出現 = 3)",     (ROOT/"default_style.py").read_text(encoding="utf-8").count("cadseq") == 3),
]
if SMOKE:
    checks.append((f"SMOKE: {MDIR}/wavs が Data/cv_r1/wavs への symlink", (ROOT/MDIR/"wavs").is_symlink()))
    checks.append(("SMOKE: config.yml の model_name = cv_r1_smoke（成果物分離）",
                   'model_name: "cv_r1_smoke"' in (ROOT/"config.yml").read_text(encoding="utf-8")))
    checks.append(("SMOKE: bf16/fp16 設定が GPU と整合", True if float(__import__("subprocess").run(
        ["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"], capture_output=True, text=True
        ).stdout.strip().splitlines()[0]) >= 8.0 else (not cfg["train"].get("bf16_run") and not cfg["train"].get("fp16_run"))))
ok = True
for name, cond in checks:
    print(("OK " if cond else "★NG"), name); ok &= bool(cond)
assert ok, "★NG を解消してから先へ（§4 / §4.5 を見直す）"
print("\n検証ゲート全通過" + ("（スモークモード）" if SMOKE else ""))


In [ ]:
# ===== §6 bert_gen（各 wav の隣に .bert.pt を生成。ローカル I/O + GPU で数分〜10分程度）=====
# Zenodo tar に .bert.pt / style npy は同梱していない → ここで生成する（Drive 直読みだと数時間かかる工程）。
!python bert_gen.py -c Data/cv_r1/config.json
import subprocess
n = int(subprocess.run("find Data/cv_r1 -name '*.bert.pt' | wc -l",
                       shell=True, capture_output=True, text=True).stdout.strip() or 0)
exp = SMOKE_EXP_TRAIN + SMOKE_EXP_VAL      # §4.5 が設定（全量 18015 / スモークはサブセット行数）
print(f"生成された .bert.pt: {n}（esd train {SMOKE_EXP_TRAIN} + val {SMOKE_EXP_VAL} = {exp} 以上が正）")
assert n >= exp, "★bert.pt 不足 → 上のログを確認（transformers が torch を見失う場合は §2 をやり直す）"


In [ ]:
# ===== §7 style_gen（スタイルベクトル生成。ローカル化で 300 it/s 級 ≈ 数分）=====
# ※ preprocess_text は走らせない（spk2id 再生成の恐れ）。style_gen は esd / config から直接生成する。
!python style_gen.py -c Data/cv_r1/config.json
import os
sv = "Data/cv_r1/style_vectors.npy"
print("style_vectors.npy:", "OK" if os.path.exists(sv)
      else "（未生成でも可: 学習開始時に default_style が生成する。§4-A のパッチ適用が前提）")


In [ ]:
# ===== §8 学習 =====
# ・全量: batch=16 / 10 epoch ≈ 8,900〜9,000 step（ローカル運用 ~3.3 it/s ≈ 2時間強、1000 step ごと保存）
# ・スモーク: 既定値で ≈ 300 step（T4 fp32 ~10分、100 step ごと保存。checkpoint は Data/cv_r1_smoke/models/）
# ・-m は「モデル出力フォルダのパス」。全量 = Data/cv_r1（★"-m cv_r1" は誤り: リポジトリ直下の
#   別ツリーに checkpoint が落ちる事故になる）/ スモーク = Data/cv_r1_smoke（§4.5 が MDIR に設定済み）
# ・開始直後のログで warm-start を必ず確認する:
#     OK → 「Loaded the pretrained models」+「Missing key: dp/sdp.cadence_cond…, emb_g.weight」群
#          （cadence_cond / emb_g は新規層なので Missing は正常）
#     NG → 「train from scratch」が出たら即中断 → §4-C（スモーク時は §4.5）の底モデル配置を確認
# ・全量時: Drive 書込で数分の「谷」が出るのは正常。途中切断したら §1→§5 を再実行してから
#   このセルを再実行（models/ の最新 checkpoint から自動再開）。スモークの ckpt は VM 揮発。
!python train_ms_jp_extra.py -c Data/cv_r1/config.json -m {MDIR}


## 出力・再開・運用ノート

**出力（いずれも実体は Drive・永続）**
- checkpoint: `Data/cv_r1/models/`（`G_*.pth / D_*.pth / WD_*.pth`、1000 step ごと）
- 推論用: `model_assets/<モデル名>/*_e10_s*.safetensors` + `style_vectors.npy`
- 総 step はバケットサンプラの丸めで 8,910〜9,019 程度に揺れる（実害なし）

**再開（切断・クラッシュ後）**
- ランタイム切断で **VM は揮発**する（torch 入替・shim・ローカルデータ・bert.pt は全消失）。
  §1→§2→§3→§4→§5 を再実行 → §6/§7 も再実行（ローカル生成物は消えている。GPU で計 ~10 分）→
  §8 を再実行すれば `models/` の最新 checkpoint から自動再開する。
- 逆に言えば**切らずに完走が得策**（ローカル運用なら実質 2 時間強で終わる）。

**Colab 運用の現実**
- 1 ランタイムで同時に動くセルは 1 つだけ。学習中の様子見・ファイル操作は**ターミナル**（別プロセス）で行う。
- pip はセッション揮発。GPU セッションのたびに §2 を再実行（clone / initialize は Drive 永続なので不要）。

**トラブル**
- 「train from scratch」→ 即中断。§4-C を確認。完全再学習をやり直す場合は `models/` を空にして
  §4-C を再実行（`*_0.safetensors` の再コピーが必要。学習が step 保存時にこれを消すのは正常挙動）。
- style 生成/学習開始直後の `ValueError: dimension 1 ... size 25 vs 40` → default_style の cadseq
  パッチ未適用（§4-A を再実行し、§5 の該当チェックが OK になることを確認）。
- bert_gen の「AutoModelForMaskedLM requires PyTorch」→ transformers が torch を見失っている。§2 をやり直す。
- pip ビルドで詰まる（`stable_ts` 等）→ §2 の除外規則 `_drop` にパッケージ名を追加（学習には不要）。
- torch DL が「止まって見える」→ §2 は進捗を流す設計（`-q` 無し）。数 GB なので数分待つ。
